In [1]:
#models
library(tidyverse)
library(caret)
library(recipes)
library(pROC)
library(dplyr)
library(tidyr)
library(ppcor)
library(readxl)


── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift



Attaching package: ‘recipes’


The following object is masked from ‘package:stringr’:

    fixed


The following object is masked from ‘package:stats’:

    step


Type 'citation("pROC")' for a citation.


Attaching packa

## Step 1: Load data

In [2]:
lucas_df_with_pcs <- read_csv("../../data/lucas_df_with_pcs.csv")
mica_df_with_pcs <- read_csv("../../data/mica_df_with_pcs.csv")
jhu_df_with_pcs <- read_csv("../../data/val_df_with_pcs.csv")

Rows: 287 Columns: 958
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (96): Patient, id, type, clinical_smokingstatus, QC, patient.type, cli...
dbl  (850): multinucratio, clinical_nlratio, clinical_CRP, clinical_cfdna_co...
lgl    (3): DIAGNOSE_OTHER_PRIMARY, Adaptor Dimer, Date Data Received
date   (9): DATE_1_VISIT_BBH, DATE_BIOPSY, DATE_SCAN_BASELINE, DATE_FEV1, DA...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 540 Columns: 1304
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (73): id, Stage, Coded Type, type, Delfi ID (Aliquot 2), DL ID, DL Suf...
dbl (1225): clinical_CCI, clinical_age, Smoking, ratio_1, ratio_2, ratio_3, ...
lgl    (6): 

In [3]:
# Build in Mica Morbidity Cohort Metadata (Annapragada et al 2026)
mica_morbidity_META <- read_excel("../../data/MICA-Morbidity-Cohort.xlsx")

# Select Sample + all cm_ columns from the metadata
cm_cols <- mica_morbidity_META %>%
  dplyr::select(Sample, starts_with("cm_"))

cm_cols

Sample,cm_Diabetes,cm_Cardiovascular disease,cm_Heart insufficient,cm_Cerebral vascular disease,cm_Dementia,cm_Connective tissue diease,cm_Mild chronic liver disease,cm_Moderate kidney failure
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DL000223NCP0,No,No,No,No,No,No,No,No
DL000224NCP0,No,No,No,No,No,No,No,No
DL000225NCP0,No,No,No,No,No,No,No,No
DL000228NCP0,No,No,No,No,No,No,No,No
DL000229NCP0,No,No,No,No,No,No,No,No
DL000230NCP0,No,No,No,No,No,No,No,No
DL000231NCP0,No,No,No,No,No,No,No,No
DL000232NCP0,No,No,No,Yes,No,Yes,No,No
DL000236NCP0,No,No,No,No,No,No,No,No


In [4]:
# Merge into mica_df_with_pcs
mica_df_with_pcs_MORBIDITY <- mica_df_with_pcs %>%
  left_join(cm_cols, by = c("id" = "Sample"))
write_csv(mica_df_with_pcs_MORBIDITY, "../../data/mica-morbidity-df.csv")

## Cross-Reactivity Plots

### Step 1: Clean and Synchronize Data

In [5]:
# ============================================================
# COHORT PREPROCESSING (LUCAS, MICA, Validation only)
# ============================================================
library(tidyverse)

# Define cm_ columns used for mica grouping
cm_cols <- grep("^cm_", names(mica_df_with_pcs_MORBIDITY), value = TRUE)

cohorts <- list(
  lucas = lucas_df_with_pcs,
  mica = mica_df_with_pcs_MORBIDITY,
  validation = jhu_df_with_pcs
)

# Quick check
#walk2(cohorts, names(cohorts), ~cat(.y, "- n:", nrow(.x), ", type:", paste(names(table(.x$type)), collapse="/"), "\n"))

# Standardize Stage: "NA" string -> actual NA
cohorts$validation <- cohorts$validation %>%
  mutate(Stage = na_if(Stage, "NA"))

# Add Cancer column
cohorts$lucas      <- cohorts$lucas      %>% mutate(Cancer = ifelse(type == "cancer", "Lung", NA_character_))
cohorts$validation <- cohorts$validation %>% mutate(Cancer = ifelse(type == "cancer", "Lung", NA_character_))
cohorts$mica       <- cohorts$mica       %>% mutate(Cancer = ifelse(type == "cancer", `Coded Type`, NA_character_))

# Helper: split cancer by stage
split_by_stage <- function(df, min_n = 5) {
  df %>%
    mutate(
      .stage_cat = case_when(
        type != "cancer" ~ NA_character_,
        Stage %in% c("I", "II") ~ "Localized",
        Stage %in% c("III", "IV") ~ "Advanced",
        TRUE ~ NA_character_
      )
    ) %>%
    group_by(Cancer) %>%
    mutate(
      n_loc = sum(.stage_cat == "Localized", na.rm = TRUE),
      n_adv = sum(.stage_cat == "Advanced", na.rm = TRUE),
      group_cancer = case_when(
        type != "cancer" | is.na(.stage_cat) ~ NA_character_,
        n_loc >= min_n & n_adv >= min_n ~ paste0(Cancer, " (", .stage_cat, ")"),
        TRUE ~ Cancer
      )
    ) %>%
    ungroup() %>%
    dplyr::select(-n_loc, -n_adv, -.stage_cat)
}

# Helper: assign mutually exclusive cm_ group for mica
assign_cm_group <- function(df, cm_cols) {
  df %>%
    rowwise() %>%
    mutate(
      .n_yes = sum(c_across(all_of(cm_cols)) == "Yes", na.rm = TRUE),
      group = case_when(
        type == "cancer"                               ~ NA_character_,
        .n_yes >= 2                                    ~ NA_character_,
        .n_yes == 0                                    ~ "Baseline healthy",
        get("cm_Diabetes") == "Yes"                    ~ "Diabetes",
        get("cm_Cardiovascular disease") == "Yes"      ~ "Cardiovascular disease",
        get("cm_Heart insufficient") == "Yes"          ~ "Heart insufficiency",
        get("cm_Cerebral vascular disease") == "Yes"   ~ "Cerebral vascular disease",
        get("cm_Dementia") == "Yes"                    ~ "Dementia",
        get("cm_Connective tissue diease") == "Yes"    ~ "Connective tissue disease",
        get("cm_Mild chronic liver disease") == "Yes"  ~ "Mild chronic liver disease",
        get("cm_Moderate kidney failure") == "Yes"     ~ "Moderate kidney failure",
        TRUE                                           ~ NA_character_
      )
    ) %>%
    ungroup() %>%
    dplyr::select(-.n_yes)
}

# Assign groups
cohorts$lucas <- cohorts$lucas %>%
  split_by_stage() %>%
  mutate(group = case_when(
    type == "cancer" ~ group_cancer,
    clinical_COPD == 1 ~ "COPD",
    grepl("asthma", clinical_chronic_inflammatory_conditions, ignore.case = TRUE) ~ "Asthma",
    grepl("rheumatoid arthritis", clinical_chronic_inflammatory_conditions, ignore.case = TRUE) ~ "Rheumatoid Arthritis",
    grepl("ulcerative colitis", clinical_chronic_inflammatory_conditions, ignore.case = TRUE) ~ "Ulcerative Colitis",
    grepl("DM 2", clinical_chronic_inflammatory_conditions, ignore.case = TRUE) ~ "DM 2",
    grepl("psoriasis", clinical_chronic_inflammatory_conditions, ignore.case = TRUE) ~ "Psoriasis",
    TRUE ~ "Baseline healthy"
  )) %>%
  dplyr::select(-group_cancer)

cohorts$mica <- cohorts$mica %>%
  split_by_stage() %>%
  assign_cm_group(cm_cols = cm_cols) %>%
  dplyr::select(-group_cancer)

cohorts$validation <- cohorts$validation %>%
  split_by_stage() %>%
  mutate(group = ifelse(type == "cancer", group_cancer, "Baseline healthy")) %>%
  dplyr::select(-group_cancer)

# Drop small groups and filter NAs
drop_small <- function(df, min_n = 4) {
  keep <- names(which(table(df$group) >= min_n))
  filter(df, group %in% keep, !is.na(group))
}
cohorts <- map(cohorts, drop_small)

# Verify
#walk2(cohorts, names(cohorts), ~{ cat("\n", .y, "groups:\n"); print(table(.x$group)) })

# Extract back to individual variables
lucas_df_with_pcs          <- cohorts$lucas
mica_df_with_pcs_MORBIDITY <- cohorts$mica
val_df_with_pcs            <- cohorts$validation

# ============================================================
# POST-PROCESSING: SAVE, RECODE STAGE LABELS
# ============================================================

common_cols <- c("id", "type", "group", "Cancer")
all_cohorts <- bind_rows(
  cohorts$lucas %>% dplyr::select(any_of(common_cols)) %>% mutate(cohort = "LUCAS"),
  cohorts$mica %>% dplyr::select(any_of(common_cols)) %>% mutate(cohort = "MICA"),
  cohorts$validation %>% dplyr::select(any_of(common_cols)) %>% mutate(cohort = "JHU")
)
write_csv(all_cohorts, "../../data/Cross-React_all_cohorts_combinedV2.csv")

# Rename lung cancer stage groups for LUCAS and Validation
cohorts$lucas <- cohorts$lucas %>%
  mutate(group = recode(group, "Lung (Localized)" = "Early (I/II)", "Lung (Advanced)" = "Late (III/IV)"))
cohorts$validation <- cohorts$validation %>%
  mutate(group = recode(group, "Lung (Localized)" = "Early (I/II)", "Lung (Advanced)" = "Late (III/IV)"))


# Update individual variables
lucas_df_with_pcs <- cohorts$lucas
val_df_with_pcs   <- cohorts$validation

# ============================================================
# RANDOM HALF-SPLIT OF NON-CANCER PATIENTS (set seed for reproducibility)
# ============================================================
set.seed(42)

# LUCAS non-cancer split
lucas_noncancer_ids <- cohorts$lucas %>% filter(type != "cancer") %>% pull(id)
lucas_split_ids     <- sample(lucas_noncancer_ids, floor(length(lucas_noncancer_ids) / 2))
lucas_split_A       <- cohorts$lucas %>% filter(id %in% lucas_split_ids)
lucas_split_B       <- cohorts$lucas %>% filter(type != "cancer" & !(id %in% lucas_split_ids))


# JHU non-cancer split
jhu_noncancer_ids <- cohorts$validation %>% filter(type != "cancer") %>% pull(id)
jhu_split_ids     <- sample(jhu_noncancer_ids, floor(length(jhu_noncancer_ids) / 2))
jhu_split_A       <- cohorts$validation %>% filter(id %in% jhu_split_ids)
jhu_split_B       <- cohorts$validation %>% filter(type != "cancer" & !(id %in% jhu_split_ids))



## Plotting

### Step 2: Fragmentome Profile Clinical Enrichment

In [6]:
# ============================================================
# THREE-TIER HIERARCHICAL DOTPLOT (MAIN FIGURE)
# ============================================================

library(tidyverse)
library(scales)

# ============================================================
# CORE STATISTICAL FUNCTIONS
# ============================================================

calc_cohens_d <- function(g1, g2) {
  g1 <- na.omit(g1); g2 <- na.omit(g2)
  if (length(g1) < 2 || length(g2) < 2) return(NA_real_)
  pooled_sd <- sqrt(((length(g1)-1)*var(g1) + (length(g2)-1)*var(g2)) / (length(g1)+length(g2)-2))
  if (pooled_sd == 0) NA_real_ else (mean(g2) - mean(g1)) / pooled_sd
}

# ============================================================
# SETUP
# ============================================================

features <- paste0("ratio_pc_", sprintf("%02d", 1:11))

# ---- PCs whose sign is flipped for display ------------------------------
# PC loadings are sign-arbitrary. Flipping selected PCs orients them
# consistently with the other panels. Because Cohen's d is a signed
# difference, this is done by negating d rather than the raw data —
# |d| (dot size) and the Wilcoxon p-value are unaffected.
INVERT_PCS      <- c(1, 2)
LABEL_INVERTED  <- TRUE      # append a marker to the inverted PC axis labels

invert_features <- paste0("ratio_pc_", sprintf("%02d", INVERT_PCS))

pc_labels <- paste0("PC", 1:11)
if (LABEL_INVERTED) {
  pc_labels[INVERT_PCS] <- paste0(pc_labels[INVERT_PCS], "\u2020")   # dagger
}

feature_labels      <- setNames(pc_labels, features)
feature_y_positions <- setNames(11:1, pc_labels)

cohort_data <- list(
  lucas      = lucas_df_with_pcs,
  mica       = mica_df_with_pcs_MORBIDITY,
  validation = val_df_with_pcs
)

cohort_display_names <- c(
  lucas      = "LUCAS Cohort",
  mica       = "MICA Morbidity Cohort",
  validation = "JHU Cohort"
)
display_to_id <- setNames(names(cohort_display_names), cohort_display_names)

condition_mapping <- list(
  "Lung cancer" = list(
    cohorts = c("LUCAS Cohort", "JHU Cohort"),
    groups = list(
      "LUCAS Cohort" = c("LUCAS Internal Control", "Early (I/II)", "Late (III/IV)"),
      "JHU Cohort"   = c("JHU Internal Control",   "Early (I/II)", "Late (III/IV)")
    )
  ),
  "Non-cancer conditions" = list(
    cohorts = c("LUCAS Cohort", "MICA Morbidity Cohort"),
    groups = list(
      "LUCAS Cohort" = c("COPD", "Asthma", "Rheumatoid Arthritis"),
      "MICA Morbidity Cohort" = c(
        "Diabetes",
        "Cardiovascular disease",
        "Heart insufficiency",
        "Cerebral vascular disease",
        "Dementia",
        "Connective tissue disease",
        "Mild chronic liver disease",
        "Moderate kidney failure"
      )
    )
  )
)

# ============================================================
# COMPUTE COHEN'S D STATISTICS
# ============================================================

ref_group <- "Baseline healthy"
all_stats <- list()

# --- LUCAS internal control: random half A vs half B ---
for (feat in features) {
  g_vals <- na.omit(lucas_split_A[[feat]])
  r_vals <- na.omit(lucas_split_B[[feat]])
  if (length(g_vals) < 2 || length(r_vals) < 2) next

  all_stats[[length(all_stats) + 1]] <- tibble(
    tier1        = "Lung cancer",
    tier2_cohort = "LUCAS Cohort",
    tier3_group  = "LUCAS Internal Control",
    feature      = feat,
    n            = length(g_vals),
    n_ref        = length(r_vals),
    cohens_d     = calc_cohens_d(r_vals, g_vals),
    p_value      = tryCatch(wilcox.test(g_vals, r_vals)$p.value, error = \(e) NA_real_)
  )
}

# --- JHU internal control: random half A vs half B ---
for (feat in features) {
  g_vals <- na.omit(jhu_split_A[[feat]])
  r_vals <- na.omit(jhu_split_B[[feat]])
  if (length(g_vals) < 2 || length(r_vals) < 2) next

  all_stats[[length(all_stats) + 1]] <- tibble(
    tier1        = "Lung cancer",
    tier2_cohort = "JHU Cohort",
    tier3_group  = "JHU Internal Control",
    feature      = feat,
    n            = length(g_vals),
    n_ref        = length(r_vals),
    cohens_d     = calc_cohens_d(r_vals, g_vals),
    p_value      = tryCatch(wilcox.test(g_vals, r_vals)$p.value, error = \(e) NA_real_)
  )
}

# --- Standard group vs baseline comparisons ---
for (t1_name in names(condition_mapping)) {
  cfg <- condition_mapping[[t1_name]]
  for (cohort_disp in cfg$cohorts) {
    cohort_df <- cohort_data[[display_to_id[[cohort_disp]]]]
    if (is.null(cohort_df)) next
    ref_df <- filter(cohort_df, group == ref_group)
    if (nrow(ref_df) == 0) next

    for (grp in cfg$groups[[cohort_disp]]) {
      # Skip internal control groups — already handled above
      if (grp %in% c("LUCAS Internal Control", "JHU Internal Control")) next
      if (!(grp %in% cohort_df$group)) next
      grp_df <- filter(cohort_df, group == grp)

      for (feat in features) {
        g_vals <- na.omit(grp_df[[feat]]); r_vals <- na.omit(ref_df[[feat]])
        if (length(g_vals) < 2 || length(r_vals) < 2) next

        all_stats[[length(all_stats) + 1]] <- tibble(
          tier1        = t1_name,
          tier2_cohort = cohort_disp,
          tier3_group  = grp,
          feature      = feat,
          n            = length(g_vals),
          n_ref        = length(r_vals),
          cohens_d     = calc_cohens_d(r_vals, g_vals),
          p_value      = tryCatch(wilcox.test(g_vals, r_vals)$p.value, error = \(e) NA_real_)
        )
      }
    }
  }
}

stats_df <- bind_rows(all_stats) %>%
  # ---- apply the display sign flip BEFORE deriving size/direction --------
  mutate(
    inverted = feature %in% invert_features,
    cohens_d = ifelse(inverted, -cohens_d, cohens_d)
  ) %>%
  mutate(
    size_d       = pmin(abs(cohens_d), 2.5),
    fill_cat     = case_when(p_value >= 0.05 ~ "ns", cohens_d > 0 ~ "higher", TRUE ~ "lower"),
    feature_disp = feature_labels[feature],
    x_id         = paste(tier2_cohort, tier3_group, sep = "___"),
    y_pos        = feature_y_positions[feature_disp]
  )


# ============================================================
# BUILD X-AXIS ORDER
# ============================================================

x_order <- c(); tier1_pos <- list(); tier2_pos <- list(); pos <- 0

for (t1_name in names(condition_mapping)) {
  cfg <- condition_mapping[[t1_name]]; t1_start <- pos + 1
  for (cohort_disp in cfg$cohorts) {
    t2_start <- pos + 1
    for (grp in cfg$groups[[cohort_disp]]) {
      x_id <- paste(cohort_disp, grp, sep = "___")
      if (x_id %in% stats_df$x_id) { pos <- pos + 1; x_order <- c(x_order, x_id) }
    }
    if (pos >= t2_start) tier2_pos[[length(tier2_pos) + 1]] <- list(name = cohort_disp, start = t2_start, end = pos, mid = (t2_start + pos) / 2)
  }
  if (pos >= t1_start) tier1_pos[[length(tier1_pos) + 1]] <- list(name = t1_name, start = t1_start, end = pos, mid = (t1_start + pos) / 2)
}

stats_df <- stats_df %>%
  filter(x_id %in% x_order) %>%
  mutate(x_id = factor(x_id, levels = x_order))

# Custom display labels
non_cancer_label_map <- c(
  "COPD"                      = "COPD",
  "Asthma"                    = "Asthma",
  "Rheumatoid Arthritis"      = "Rheumatoid\nArthritis",
  "Diabetes"                  = "Diabetes",
  "Cardiovascular disease"    = "Cardiovascular\nDisease",
  "Heart insufficiency"       = "Heart\nInsufficiency",
  "Cerebral vascular disease" = "Cerebral\nVascular\nDisease",
  "Dementia"                  = "Dementia",
  "Connective tissue disease" = "Connective\nTissue\nDisease",
  "Mild chronic liver disease"= "Mild Chronic\nLiver Disease",
  "Moderate kidney failure"   = "Moderate\nKidney Failure"
)

x_labels <- stats_df %>%
  group_by(x_id) %>%
  summarise(
    tier2  = first(tier2_cohort),
    tier3  = first(tier3_group),
    n_val  = first(n),
    n_ref  = first(n_ref),
    .groups = "drop"
  ) %>%
  mutate(label = case_when(
    # Internal controls: show both half sizes
    tier3 == "LUCAS Internal Control" ~ paste0("Non-cancer\n(n=158)"),
    tier3 == "JHU Internal Control"   ~ paste0("Non-cancer\n(n=385)"),
    # Lung cancer stage: shortened label with n
    tier3 == "Early (I/II)"  ~ paste0("I/II\n(n=", n_val, ")"),
    tier3 == "Late (III/IV)" ~ paste0("III/IV\n(n=", n_val, ")"),
    # Non-cancer: apply line-break map, append n
    tier3 %in% names(non_cancer_label_map) ~
      paste0(non_cancer_label_map[tier3], "\n(n=", n_val, ")"),
    # Fallback
    TRUE ~ paste0(tier3, "\n(n=", n_val, ")")
  )) %>%
  dplyr::select(x_id, label) %>%
  deframe()

# ============================================================
# MAIN DOTPLOT (STANDALONE)
# ============================================================

fill_cols <- c(lower = "#8E44AD", higher = "#E67E22", ns = "#CCCCCC")

p_main <- ggplot(stats_df, aes(x = x_id, y = y_pos)) +
  geom_hline(yintercept = 1:11, color = "grey92", linewidth = 0.3) +
  geom_point(aes(size = size_d, fill = fill_cat), shape = 21, color = "black", stroke = 0.5) +
  scale_x_discrete(labels = x_labels) +
  scale_y_continuous(breaks = feature_y_positions, labels = names(feature_y_positions), limits = c(0.5, 15)) +
  scale_fill_manual(values = fill_cols, name = "Direction",
    labels = c(higher = "Higher than reference", lower = "Lower than reference", ns = "Not significant")) +
  scale_size_continuous(range = c(4, 12), limits = c(0, 2.5), breaks = c(0.5, 1, 1.5, 2), name = "|Cohen's d|") +
  labs(x = NULL, y = NULL,
       caption = if (LABEL_INVERTED)
           paste0("")
         else NULL) +
  theme_minimal(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 0, hjust = 0.5, vjust = 1, size = 18),
    axis.text.y = element_text(size = 20, color = "black"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    legend.position = "bottom",
    legend.direction = "horizontal",
    legend.box = "horizontal",
    plot.caption = element_text(size = 14, hjust = 0, color = "grey30")
  ) +
  guides(
    fill = guide_legend(override.aes = list(size = 5), order = 1, nrow = 1),
    size = guide_legend(order = 2, nrow = 1)
  ) +
  coord_cartesian(clip = "off")

# Add tier2 annotations (cohort brackets)
for (t2 in tier2_pos) p_main <- p_main +
  annotate("segment", x = t2$start - 0.4, xend = t2$end + 0.4, y = 12, yend = 12, color = "grey40") +
  annotate("text", x = t2$mid, y = 12.5, label = t2$name, hjust = 0.5, size = 7)

# Add tier1 annotations (condition brackets)
for (t1 in tier1_pos) p_main <- p_main +
  annotate("segment", x = t1$start - 0.4, xend = t1$end + 0.4, y = 13.5, yend = 13.5, linewidth = 1) +
  annotate("text", x = t1$mid, y = 14, label = t1$name, hjust = 0.5, size = 9)

# Vertical separator between tier1 groups
for (i in seq_along(tier1_pos)[-length(tier1_pos)]) {
  p_main <- p_main + annotate("segment",
    x = tier1_pos[[i]]$end + 0.5, xend = tier1_pos[[i]]$end + 0.5,
    y = 0.5, yend = 11.5, color = "grey50", linewidth = 0.8, linetype = "dashed")
}

# Print
options(repr.plot.width = 22, repr.plot.height = 15, repr.plot.res = 500)
print(p_main)


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor




In [7]:
ggsave("../../outputs/Fig5/Fig-Clinical-Enrichment.png", plot = p_main, 
       width = 22, height = 16, dpi = 600, units = "in", create.dir = TRUE)

In [8]:
# Save the stats dataframe used for the dotplot
#write_csv(stats_df, "dotplot_cohens_d_stats.csv")
#stats_df

stats_df_clean <- stats_df %>%
  dplyr::select(
    Tier1 = tier1,
    Cohort = tier2_cohort,
    Group = tier3_group,
    Feature = feature,
    PC = feature_disp,
    N = n,
    Cohens_d = cohens_d,
    P_value = p_value,
    Direction = fill_cat
  )

write_csv(stats_df_clean, "../../data/Clinical-Enrichment-dotplot_cohens_d_stats_cleanV2.csv")

In [ ]:
# Done #